# Prompting Notebook

## Import libraries

In [ ]:
#  Library imports
import os
import sys
from pathlib import Path
from openai import AsyncOpenAI
#from dotenv import load_dotenv

# Now add parent to sys.path
#parent_dir = Path.cwd().parent
#sys.path.append(str(parent_dir))

# Construct full path to .env in the project root
#env_path = Path("..").resolve().parent / "ommitted"

# Load .env file
#load_dotenv(dotenv_path=env_path)

### API KEY PLACEHOLDER ###

# Source code imports
from src.prompt_pipeline import LlmSurveyParser
from src.build_survey_dict import build_survey_dict
from src.multi_llm_call_OPENAI import LLMOpenAIClient
from src.JSON_creator import save_survey_results
from src.extract_output import extract

## Parameterization 

#### LLM Model Selection:
##### Open Ai
- gpt-5-nano 
    - reasoning: (minimal / high)

##### Deepseek:
- deepseek-ai/DeepSeek-V3.2 

##### mistralai:
- mistralai/Mistral-Small-24B-Instruct-2501

##### meta-llama:
- meta-llama/Meta-Llama-3.1-8B-Instruct
  
#### survey list: 
- science_direct
- AFSP
- duke
- ipsos
- progress
- YouGOV_Biden_Trump
- YouGOV_Suicide_prevention

### Please note: each survey is done separately, the example below is Science Direct Q3

In [244]:
param = {"model" : "deepseek-ai/DeepSeek-V3.2",
        "concurrency" : 200,
        "survey" : "science_direct",
        "reasoning_effort": "none",
         "temperature": 1.0,
         "top_p": 0.95
        }

In [245]:
survey_data = build_survey_dict(excel_path = "../../data/S1 Survey Data.xlsx", sheet_name = param.get("survey"))

### Generate prompts

In [246]:
direct_role_parser = LlmSurveyParser(dictionary=survey_data, subgroups="false", embody="false")
direct_all_info = direct_role_parser.run_all()
direct_prompts = direct_role_parser.get_prompt_texts()


embodied_role_parser = LlmSurveyParser(dictionary=survey_data, subgroups="true", embody="true")
embodied_all_info = embodied_role_parser.run_all()
embodied_prompts = embodied_role_parser.get_prompt_texts()

expert_evaluator_parser = LlmSurveyParser(dictionary=survey_data, subgroups="true", embody="false")
expert_all_info = expert_evaluator_parser.run_all()
expert_evaluator_prompts = expert_evaluator_parser.get_prompt_texts()

Generated 5 prompts.
Generated 6480 prompts.
Generated 6480 prompts.


### Sample prompts

In [247]:
print(direct_prompts[0])
print()
print(embodied_prompts[0])
print()
print(expert_evaluator_prompts[0])

[{'role': 'user', 'content': 'You are a public opinion expert. You are presented with a survey question asking how a US citizen would respond with one of the following responses. The question you are answering is: “988” was nationally launched as the three-digit dialing code for the 988 Suicide & Crisis Lifeline in July 2022. Please rate, on a scale of 1-7, the likelihood of you reaching out to each of the following sources if you or a loved one were experiencing a mental health crisis or thoughts of suicide (1= Very unlikely, 7= Very likely)? · 988 Lifeline Select a response option from the following: [1, 2, 3, 4, 5, 6, or 7.] Please choose from the options provided and then give a very brief reason why. The format should be: {selected option}, brief reasoning.'}]

[{'role': 'user', 'content': 'You are a non-Hispanic White, identifying as female for gender, aged between 18 and 29, do not have a high school diploma, earning less than $50K, identifying as a Democrat voter, living in the

### Generate Dictionary for LLM input

In [248]:
def get_all_sets(all_info, all_prompts, title):
    results = []
    for info, prompt in zip(all_info, all_prompts): # put them together
        if info["title"] == title: # search by title 
            results.append((info, prompt))
    return results

info_prompt_dictionary = {}
SD_titles = [direct_all_info[index].get('title') for index in range(len(direct_all_info))]
direct_sets = [get_all_sets(direct_all_info, direct_prompts, title) for title in SD_titles]
embodied_sets = [get_all_sets(embodied_all_info, embodied_prompts, title) for title in SD_titles]
expert_sets = [get_all_sets(expert_all_info, expert_evaluator_prompts, title) for title in SD_titles]

for index in range(len(SD_titles)):
    info_prompt_dictionary[f"direct_info_Q{index+1}"] = [info for info, prompt in direct_sets[index]]
    info_prompt_dictionary[f"direct_prompt_Q{index+1}"] = [prompt for info, prompt in direct_sets[index]]
    info_prompt_dictionary[f"embodied_info_Q{index+1}"] = [info for info, prompt in embodied_sets[index]]
    info_prompt_dictionary[f"embodied_prompt_Q{index+1}"] = [prompt for info, prompt in embodied_sets[index]]
    info_prompt_dictionary[f"specialist_info_Q{index+1}"] = [info for info, prompt in expert_sets[index]]
    info_prompt_dictionary[f"specialist_prompt_Q{index+1}"] = [prompt for info, prompt in expert_sets[index]]

### Initiate LLM Client

In [ ]:
def init_llm_client(model: str, concurrency: int):
    
    # If the model name starts with "gpt", assume it is an OpenAI model
    # and initialize the standard OpenAI async client
    if model.startswith("gpt"):
        client = AsyncOpenAI(api_key=openai_api_key) #initialize
        return LLMOpenAIClient(client=client, concurrency=concurrency)

    # If the model belongs to Meta, DeepSeek, or Mistral families, request through the DeepInfra
    if model.startswith(("meta", "deepseek","mistralai")):
        
        # Base URL for DeepInfra's open ai compatible API
        base_url = "https://api.deepinfra.com/v1/openai" 

        api_key = deepinfra_api_key
          
        client = AsyncOpenAI(api_key=api_key, base_url=base_url)  # initialize
        return LLMOpenAIClient(client=client, concurrency=concurrency)

    # Raise an error if provided is not recognized
    raise ValueError(f"Unknown model: {model}")

llm_client = init_llm_client(param.get("model"), param.get("concurrency"))

### Set LLM Provider

In [ ]:
# Map providers to the models they support

MODEL_ROUTER = {
    "openai": (
        "gpt-5-nano",
        "deepseek-ai/DeepSeek-V3.2",
        "meta-llama/Meta-Llama-3.1-8B-Instruct",
        "mistralai/Mistral-Small-24B-Instruct-2501"
    )
}

def get_provider(model):
    # Iterate through the router to find which provider the model belongs to
    for provider, model_names in MODEL_ROUTER.items():
        if model in model_names:
            return provider  # Return the matching provider
    return None  # If the model is not found in the router

provider = get_provider(param.get("model"))

### Set the question and iteration number

In [253]:
#survey question number
q = "Q3"

#iteration number
it = 2

### Create folder for results

In [254]:
#create file path to result folder to store LLM output
if param.get("reasoning_effort") == "none":
    folder_path = rf"../../results/02_final_testing_results/{param.get("survey")}/{q}/{param.get("model")}"
else: 
    folder_path = rf"../../results/02_final_testing_results/{param.get("survey")}/{q}/{param.get("model")}/{param.get("reasoning_effort")}"

#create result folder if folder doesn't exist
os.makedirs(folder_path, exist_ok=True)

### Generate Direct Responses

In [ ]:
file_path = f"{folder_path}/raw_direct_{param.get("survey")}_{q}.json"

async def main():
    # OpenAI / DeepSeek / GPT models
    if provider == "openai":

         results = await llm_client.prompting_process(messages_list= info_prompt_dictionary[f'direct_prompt_{q}'],
                                                      model= param.get("model"),
                                                       top_p = param.get("top_p"),
                                                      reasoning_effort= param.get("reasoning_effort"),
                                                    temperature =  param.get("temperature")
                                                            
                                                     )

    else:
        raise ValueError(f"Unknown provider for model: {model}")

    save_survey_results(
        info_prompt_dictionary[f'direct_info_{q}'],
        results,
        param.get("model"),
        it,
        file_path,
        param.get("reasoning_effort")
    )

await main()

100%|██████████| 1/1 [00:02<00:00,  2.62s/it]

Saved 1 rows to ../../results/02_final_testing_results/science_direct/Q3/deepseek-ai/DeepSeek-V3.2/raw_direct_science_direct_Q3.json as JSON


### Generate Embody Responses

In [256]:
file_path = f"{folder_path}/raw_embody_{param.get("survey")}_{q}.json"

async def main():
    # OpenAI / DeepSeek / GPT models
    if provider == "openai":
        results = await llm_client.prompting_process(messages_list= info_prompt_dictionary[f'embodied_prompt_{q}'],
                                                      model= param.get("model"), 
                                                      reasoning_effort= param.get("reasoning_effort"),
                                                     temperature =  param.get("temperature"),
                                                      top_p = param.get("top_p")
                                                    )

    else:
        raise ValueError(f"Unknown provider for model: {model}")

    save_survey_results(
        info_prompt_dictionary[f'embodied_info_{q}'],
        results,
        param.get("model"),
        it,
        file_path,
        param.get("reasoning_effort")
    )

await main()

100%|██████████| 1296/1296 [01:05<00:00, 19.76it/s]

Saved 1296 rows to ../../results/02_final_testing_results/science_direct/Q3/deepseek-ai/DeepSeek-V3.2/raw_embody_science_direct_Q3.json as JSON


### Generate Specialist Responses

In [257]:
file_path = f"{folder_path}/raw_specialist_{param.get("survey")}_{q}.json"

async def main():
    # OpenAI / DeepSeek / GPT models
    if provider == "openai":
        results = await llm_client.prompting_process(messages_list= info_prompt_dictionary[f'specialist_prompt_{q}'],
                                                      model= param.get("model"), 
                                                      reasoning_effort= param.get("reasoning_effort"),
                                                     temperature =  param.get("temperature"),
                                                      top_p = param.get("top_p")
                                                    )


    else:
        raise ValueError(f"Unknown provider for model: {model}")

    save_survey_results(
        info_prompt_dictionary[f'specialist_info_{q}'],
        results,
        param.get("model"),
        it,
        file_path,
        param.get("reasoning_effort")
    )

await main()

100%|██████████| 1296/1296 [00:59<00:00, 21.62it/s]

Saved 1296 rows to ../../results/02_final_testing_results/science_direct/Q3/deepseek-ai/DeepSeek-V3.2/raw_specialist_science_direct_Q3.json as JSON


## Generate Analyzed Reponse JSON

In [258]:
json_files = [
    f for f in os.listdir(folder_path)
    if f.endswith(".json")
]

In [259]:
for file in json_files:
    if file.startswith('raw'):
        file_path = f"{folder_path}/{file}"
        extract(file_path)